# Triton Kernel Profiling on AMD GPUs

## Get the System Hardware Info

In [1]:
!amd-smi

+------------------------------------------------------------------------------+
| AMD-SMI 26.0.2+39589fda      amdgpu version: 6.16.13  ROCm version: 7.0.3    |
| Platform: Linux Guest                                                        |
|-------------------------------------+----------------------------------------|
| BDF                        GPU-Name | Mem-Uti   Temp   UEC       Power-Usage |
| GPU  HIP-ID  OAM-ID  Partition-Mode | GFX-Uti    Fan               Mem-Usage |
|=====================================+========================================|
| 0000:ff:00.0 AMD Instinct MI300X VF | N/A        N/A   0           N/A/750 W |
|   0       0       7        SPX/NPS1 | N/A        N/A           285/196288 MB |
+-------------------------------------+----------------------------------------+
+------------------------------------------------------------------------------+
| Processes:                                                                   |
|  GPU        PID  Process N

## Check profiling environment

In [2]:
!rocprof-compute --version

----------------------------------------
rocprofiler-compute version: 3.2.3 (release)
Git revision:     1d722c14
----------------------------------------


## Triton MatMulBias Kernel Program

In [3]:
%%writefile  triton/matmulbias.py
import argparse
import torch

import triton
import triton.language as tl
import triton.profiler as proton

def _matmul_launch_metadata(grid, kernel, args):
    ret = {}
    M, N, K, WS = args["M"], args["N"], args["K"], args.get("WARP_SPECIALIZE", False)
    BM, BN, BK = args["BLOCK_SIZE_M"], args["BLOCK_SIZE_N"], args["BLOCK_SIZE_K"]
    ws_str = "_ws" if WS else ""
    ret["name"] = f"{kernel.name}{ws_str} [M={M}, N={N}, K={K}] [BM={BM}, BN={BN} BK={BK}]"
    if "output_ptr" in args:
        bytes_per_elem = args["output_ptr"].element_size()
    else:
        bytes_per_elem = 2
    ret[f"flops{bytes_per_elem * 8}"] = 2. * M * N * K
    ret["bytes"] = bytes_per_elem * (M * K + N * K + M * N)
    return ret

def _bias_launch_metadata(grid, kernel, args):
    ret = {}
    M, N, WS = args["M"], args["N"], args.get("WARP_SPECIALIZE", False)
    BM, BN = args["BLOCK_SIZE_M"], args["BLOCK_SIZE_N"]
    ws_str = "_ws" if WS else ""
    ret["name"] = f"{kernel.name}{ws_str} [M={M}, N={N}] [BM={BM}, BN={BN}]"
    if "output_ptr" in args:
        bytes_per_elem = args["output_ptr"].element_size()
    else:
        bytes_per_elem = 2
    ret[f"flops{bytes_per_elem * 8}"] = 2. * M * N
    ret["bytes"] = bytes_per_elem * (M + N + M * N)
    return ret


def matmul_autotune_config(pre_hook=None):
    return [
        triton.Config({'BLOCK_SIZE_M': BM, 'BLOCK_SIZE_N': BN, "BLOCK_SIZE_K": BK, "GROUP_SIZE_M": 8}, num_stages=s,
            num_warps=w, pre_hook=pre_hook)
        for BM in [128]
        for BN in [128, 256]
        for BK in [64, 128]
        for s in ([3, 4, 5])
        for w in [4, 8]
    ]

def bias_autotune_config(pre_hook=None):
    return [
        triton.Config({'BLOCK_SIZE_M': BM, 'BLOCK_SIZE_N': BN, "GROUP_SIZE_M": 8}, num_stages=s,
            num_warps=w, pre_hook=pre_hook)
        for BM in [128]
        for BN in [128, 256]
        for s in ([3, 4, 5])
        for w in [4, 8]
    ]

# MatMul kernel
@triton.autotune(
    configs=matmul_autotune_config(),
    key=['M', 'N', 'K'],
)
@triton.jit(launch_metadata=_matmul_launch_metadata)
def matmul_kernel(
        a_ptr, b_ptr, c_ptr,
        M, N, K,
        stride_am, stride_ak,
        stride_bk, stride_bn,
        stride_cm, stride_cn,
        BLOCK_SIZE_M: tl.constexpr, BLOCK_SIZE_N: tl.constexpr, BLOCK_SIZE_K: tl.constexpr,
        GROUP_SIZE_M: tl.constexpr
):
    pid = tl.program_id(axis=0)
    num_pid_m = tl.cdiv(M, BLOCK_SIZE_M)
    num_pid_n = tl.cdiv(N, BLOCK_SIZE_N)
    num_pid_in_group = GROUP_SIZE_M * num_pid_n
    group_id = pid // num_pid_in_group
    first_pid_m = group_id * GROUP_SIZE_M
    group_size_m = min(num_pid_m - first_pid_m, GROUP_SIZE_M)
    pid_m = first_pid_m + ((pid % num_pid_in_group) % group_size_m)
    pid_n = (pid % num_pid_in_group) // group_size_m

    tl.assume(pid_m >= 0)
    tl.assume(pid_n >= 0)
    tl.assume(stride_am > 0)
    tl.assume(stride_ak > 0)
    tl.assume(stride_bn > 0)
    tl.assume(stride_bk > 0)
    tl.assume(stride_cm > 0)
    tl.assume(stride_cn > 0)

    offs_am = (pid_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)) % M
    offs_bn = (pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)) % N
    offs_k = tl.arange(0, BLOCK_SIZE_K)
    a_ptrs = a_ptr + (offs_am[:, None] * stride_am + offs_k[None, :] * stride_ak)
    b_ptrs = b_ptr + (offs_k[:, None] * stride_bk + offs_bn[None, :] * stride_bn)

    accumulator = tl.zeros((BLOCK_SIZE_M, BLOCK_SIZE_N), dtype=tl.float32)
    for k in range(0, tl.cdiv(K, BLOCK_SIZE_K)):
        a = tl.load(a_ptrs, mask=offs_k[None, :] < K - k * BLOCK_SIZE_K, other=0.0)
        b = tl.load(b_ptrs, mask=offs_k[:, None] < K - k * BLOCK_SIZE_K, other=0.0)
        accumulator = tl.dot(a, b, accumulator)
        a_ptrs += BLOCK_SIZE_K * stride_ak
        b_ptrs += BLOCK_SIZE_K * stride_bk
    c = accumulator.to(tl.float16)

    offs_cm = pid_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)
    offs_cn = pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)
    c_ptrs = c_ptr + stride_cm * offs_cm[:, None] + stride_cn * offs_cn[None, :]
    c_mask = (offs_cm[:, None] < M) & (offs_cn[None, :] < N)
    tl.store(c_ptrs, c, mask=c_mask)

# Bias kernel
@triton.autotune(
    configs=bias_autotune_config(),
    key=['M', 'K'],
)
@triton.jit(launch_metadata=_bias_launch_metadata)
def bias_kernel(
        a_ptr,
        bias_ptr,
        output_ptr,
        M, N,
        stride_m, stride_n,
        BLOCK_SIZE_M: tl.constexpr, BLOCK_SIZE_N: tl.constexpr, GROUP_SIZE_M: tl.constexpr,
):
    pid = tl.program_id(axis=0)
    num_pid_m = tl.cdiv(M, BLOCK_SIZE_M)
    num_pid_n = tl.cdiv(N, BLOCK_SIZE_N)
    num_pid_in_group = GROUP_SIZE_M * num_pid_n
    group_id = pid // num_pid_in_group
    first_pid_m = group_id * GROUP_SIZE_M
    group_size_m = min(num_pid_m - first_pid_m, GROUP_SIZE_M)
    pid_m = first_pid_m + ((pid % num_pid_in_group) % group_size_m)
    pid_n = (pid % num_pid_in_group) // group_size_m

    offs_m = (pid_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)) % M
    offs_n = (pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)) % N
    a_ptrs = a_ptr + (offs_m[:, None] * stride_m + offs_n[None, :] * stride_n)
    
    a = tl.load(a_ptrs)
    
    bias = tl.load(bias_ptr + offs_n, mask=offs_n < N, other=0.0)

    output = a + bias[None, :]
        
    offs_m = pid_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)
    offs_n = pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)
    output_ptrs = output_ptr + stride_m * offs_m[:, None] + stride_n * offs_n[None, :]
    output_mask = (offs_m[:, None] < M) & (offs_n[None, :] < N)
    tl.store(output_ptrs, output, mask=output_mask)

# MatMulBias Fusion kernel
@triton.autotune(
    configs=matmul_autotune_config(),
    key=['M', 'N', 'K'],
)
@triton.jit(launch_metadata=_matmul_launch_metadata)
def matmulbias_kernel(
        a_ptr, b_ptr, c_ptr,
        bias_ptr,
        M, N, K,
        stride_am, stride_ak,
        stride_bk, stride_bn,
        stride_cm, stride_cn,
        BLOCK_SIZE_M: tl.constexpr, BLOCK_SIZE_N: tl.constexpr, BLOCK_SIZE_K: tl.constexpr,
        GROUP_SIZE_M: tl.constexpr
):
    pid = tl.program_id(axis=0)
    num_pid_m = tl.cdiv(M, BLOCK_SIZE_M)
    num_pid_n = tl.cdiv(N, BLOCK_SIZE_N)
    num_pid_in_group = GROUP_SIZE_M * num_pid_n
    group_id = pid // num_pid_in_group
    first_pid_m = group_id * GROUP_SIZE_M
    group_size_m = min(num_pid_m - first_pid_m, GROUP_SIZE_M)
    pid_m = first_pid_m + ((pid % num_pid_in_group) % group_size_m)
    pid_n = (pid % num_pid_in_group) // group_size_m

    tl.assume(pid_m >= 0)
    tl.assume(pid_n >= 0)
    tl.assume(stride_am > 0)
    tl.assume(stride_ak > 0)
    tl.assume(stride_bn > 0)
    tl.assume(stride_bk > 0)
    tl.assume(stride_cm > 0)
    tl.assume(stride_cn > 0)

    offs_am = (pid_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)) % M
    offs_bn = (pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)) % N
    offs_k = tl.arange(0, BLOCK_SIZE_K)
    a_ptrs = a_ptr + (offs_am[:, None] * stride_am + offs_k[None, :] * stride_ak)
    b_ptrs = b_ptr + (offs_k[:, None] * stride_bk + offs_bn[None, :] * stride_bn)

    accumulator = tl.zeros((BLOCK_SIZE_M, BLOCK_SIZE_N), dtype=tl.float32)
    for k in range(0, tl.cdiv(K, BLOCK_SIZE_K)):
        a = tl.load(a_ptrs, mask=offs_k[None, :] < K - k * BLOCK_SIZE_K, other=0.0)
        b = tl.load(b_ptrs, mask=offs_k[:, None] < K - k * BLOCK_SIZE_K, other=0.0)
        accumulator = tl.dot(a, b, accumulator)
        a_ptrs += BLOCK_SIZE_K * stride_ak
        b_ptrs += BLOCK_SIZE_K * stride_bk
    c = accumulator.to(tl.float16)

    bias = tl.load(bias_ptr + offs_bn, mask=offs_bn < N, other=0.0)
    output = c + bias[None, :]

    offs_cm = pid_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)
    offs_cn = pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)
    c_ptrs = c_ptr + stride_cm * offs_cm[:, None] + stride_cn * offs_cn[None, :]
    c_mask = (offs_cm[:, None] < M) & (offs_cn[None, :] < N)
    tl.store(c_ptrs, output, mask=c_mask)


# MatMulBias kernel wrapper function
def nonfusion_matmulbias(a: torch.Tensor, b: torch.Tensor, bias: torch.Tensor):
    assert a.shape[1] == b.shape[0], "Incompatible dimensions"
    assert a.is_contiguous(), "Matrix A must be contiguous"
    M, K = a.shape
    K, N = b.shape
    assert bias.shape[0] == N, "BIAS has incompatible dimensions"
    c = torch.empty((M, N), device=a.device, dtype=torch.float16)
    matmul_grid = lambda META: (triton.cdiv(M, META['BLOCK_SIZE_M']) * triton.cdiv(N, META['BLOCK_SIZE_N']), )
    matmul_kernel[matmul_grid](
        a, b, c,
        M, N, K,
        a.stride(0), a.stride(1),
        b.stride(0), b.stride(1),
        c.stride(0), c.stride(1),
    )

    o = torch.empty((M, N), device=c.device, dtype=torch.float16)
    bias_grid = lambda META: (triton.cdiv(M, META['BLOCK_SIZE_M']) * triton.cdiv(N, META['BLOCK_SIZE_N']), )
    bias_kernel[bias_grid](
        c, bias, o,
        M, N,
        c.stride(0), c.stride(1),
    )
    
    return o

# MatMulBias kernel wrapper function
def fusion_matmulbias(a: torch.Tensor, b: torch.Tensor, bias: torch.Tensor):
    assert a.shape[1] == b.shape[0], "Incompatible dimensions"
    assert a.is_contiguous(), "Matrix A must be contiguous"
    M, K = a.shape
    K, N = b.shape
    assert bias.shape[0] == N, "BIAS has incompatible dimensions"
    o = torch.empty((M, N), device=a.device, dtype=torch.float16)
    grid = lambda META: (triton.cdiv(M, META['BLOCK_SIZE_M']) * triton.cdiv(N, META['BLOCK_SIZE_N']), )
    matmulbias_kernel[grid](
        a, b, o,
        bias,
        M, N, K,
        a.stride(0), a.stride(1),
        b.stride(0), b.stride(1),
        o.stride(0), o.stride(1),
    )
    
    return o


def get_rtol():
    target = triton.runtime.driver.active.get_current_target()
    if target.backend == "hip":
        if target.arch == "gfx90a":
            return 1e-2
        elif target.arch == "gfx942":
            return 1e-2
    else:
        return 0

def show_profile(profile_name):
    import triton.profiler.viewer as proton_viewer
    metric_names = ["time/ms", "flop16/s"]
    file_name = f"profiles/{profile_name}.hatchet"
    tree, metrics = proton_viewer.parse(metric_names, file_name)

    print(f"Proton profile results for {profile_name}")
    proton_viewer.print_tree(tree, metrics)


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--profile", action="store_true", help="run the proton profiler")
    parser.add_argument("--trace", action="store_true", help="enable python tracing for the proton profiler")
    parser.add_argument("--verify", action="store_true", help="run a torch version of the kernel and compare the results")
    parser.add_argument("--M", default=1024, type=int, help="M size of the input matrices, default: %(default)d")
    parser.add_argument("--N", default=1024, type=int, help="N size of the input matrices, default: %(default)d")
    parser.add_argument("kernel", choices=["fusion", "nonfusion"], help="select which version of the kernel to run")
    args = parser.parse_args()

    # Test matrices
    torch.manual_seed(0)
    M = args.M
    N = args.N
    a = torch.randn((M, N), device='cuda', dtype=torch.float16)
    b = torch.randn((N, M), device='cuda', dtype=torch.float16)
    bias = torch.randn(N, device='cuda', dtype=torch.float16)

    if args.profile:
        if args.trace:
            context = "python"
        else:
            context = "shadow"

        print(f"Profiling the {args.kernel} matmulbias kernel (input size M:{args.M} N:{args.N}) ...")
        proton.start(f"profiles/{args.kernel}_matmulbias_{args.M}x{args.N}", context=context, hook="triton")
    else:
        print(f"Running the {args.kernel} matmulbias kernel (input size M:{args.M} N:{args.N}) ...")
    
    if args.kernel == "fusion":
        triton_output = fusion_matmulbias(a, b, bias)
    elif args.kernel == "nonfusion":
        triton_output = nonfusion_matmulbias(a, b, bias)

    if args.profile:
        proton.finalize()
    print(f"triton_output: {triton_output}")

    if args.profile:
        show_profile(f"{args.kernel}_matmulbias_{args.M}x{args.N}")

    if args.verify:
        print(f"Running the torch matmulbias kernel ...")
        torch_output = torch.addmm(bias, a, b)
        print(f"torch_output: {torch_output}")
        print("Verifying triton results with torch ...")
        if torch.allclose(triton_output, torch_output, atol=1e-2, rtol=get_rtol()):
            print("Verified")
        else:
            print("Differ")


Overwriting triton/matmulbias.py


### Help

In [4]:
!python triton/matmulbias.py --help

usage: matmulbias.py [-h] [--profile] [--trace] [--verify] [--M M] [--N N]
                     {fusion,nonfusion}

positional arguments:
  {fusion,nonfusion}  select which version of the kernel to run

options:
  -h, --help          show this help message and exit
  --profile           run the proton profiler
  --trace             enable python tracing for the proton profiler
  --verify            run a torch version of the kernel and compare the
                      results
  --M M               M size of the input matrices, default: 1024
  --N N               N size of the input matrices, default: 1024


## Non-Fusion MatMulBias Kernel

### Verify the Non-Fusion MatMulBias Kernel
This also ensures the kernel has been built and cached. The results from the triton kernel are compared against torch.

#### Small Input
Default size is M: 1024 and N: 1024

In [5]:
!python triton/matmulbias.py --verify nonfusion

Running the nonfusion matmulbias kernel (input size M:1024 N:1024) ...
triton_output: tensor([[  2.7480,  43.6250,  30.9531,  ...,  26.2500,   8.8203,   6.4609],
        [  7.1562, -21.3906,  17.7969,  ..., -16.8125,  29.6562, -48.3125],
        [ 13.8125,  -6.4609,   0.5146,  ...,  48.0000,  25.1562, -27.5156],
        ...,
        [-54.4375,  -1.4375,  -0.4214,  ...,   5.4297,  17.2812, -39.0625],
        [  9.6094, -52.2500,  43.0000,  ..., -45.8438, -49.3125,  32.6875],
        [-12.8516,   9.5391,  53.3750,  ...,  -9.3438,   7.6797,  54.1250]],
       device='cuda:0', dtype=torch.float16)
Running the torch matmulbias kernel ...
torch_output: tensor([[  2.7480,  43.6562,  30.9531,  ...,  26.2500,   8.8203,   6.4609],
        [  7.1602, -21.3906,  17.7969,  ..., -16.8125,  29.6406, -48.3125],
        [ 13.8125,  -6.4609,   0.5142,  ...,  48.0312,  25.1406, -27.5156],
        ...,
        [-54.4375,  -1.4375,  -0.4214,  ...,   5.4297,  17.2812, -39.0625],
        [  9.6094, -52.2500,

#### Large Input

In [6]:
!python triton/matmulbias.py --M 32768 --N 32768 --verify nonfusion

Running the nonfusion matmulbias kernel (input size M:32768 N:32768) ...
triton_output: tensor([[ -60.8750,   56.6562,   57.0938,  ...,  274.0000,  295.2500,
           14.8828],
        [ 205.3750, -353.7500, -102.1875,  ...,  -24.4219,   57.0625,
          -30.7969],
        [  56.2188,  258.7500,   46.6562,  ..., -366.0000,  395.2500,
          285.2500],
        ...,
        [  -5.0977,   99.9375,  -73.5625,  ...,  -70.5000,  204.2500,
         -156.8750],
        [  86.8125,  -26.3594, -128.8750,  ...,  -60.7500,  -36.3750,
          100.8750],
        [ -27.6406,   93.5625,   77.9375,  ..., -213.1250, -323.2500,
          -16.8438]], device='cuda:0', dtype=torch.float16)
Running the torch matmulbias kernel ...
torch_output: tensor([[ -60.8750,   56.6562,   57.1250,  ...,  274.0000,  295.2500,
           14.8828],
        [ 205.3750, -353.7500, -102.1875,  ...,  -24.4219,   57.0312,
          -30.7969],
        [  56.2188,  258.7500,   46.6562,  ..., -366.0000,  395.2500,
        

### Profile the Non-Fusion MatMulBias Kernel

#### Using the Triton Proton CLI
Generates a hatchet file, nonfusion_matmulbias_<M>x<N>.hatchet, under the profiles directory.

##### Help

In [7]:
!proton --help

usage: 
    proton [options] script.py [script_args] [script_options]
    proton [options] pytest [pytest_args] [script_options]
    python -m triton.profiler.proton [options] script.py [script_args] [script_options]

The proton command utility for profiling scripts and pytest tests.

positional arguments:
  target_args           Subcommand and its arguments

options:
  -h, --help            show this help message and exit
  -n NAME, --name NAME  Name of the profiling session
  -b {cupti,roctracer,instrumentation}, --backend {cupti,roctracer,instrumentation}
                        Profiling backend
  -c {shadow,python}, --context {shadow,python}
                        Profiling context
  -m MODE, --mode MODE  Profiling mode
  -d {tree,trace}, --data {tree,trace}
                        Profiling data
  -k {None,triton}, --hook {None,triton}
                        Profiling hook


##### Small Input

In [8]:
!proton -d tree -n profiles/nonfusion_matmulbias_1024x1024 -k triton triton/matmulbias.py nonfusion

Running the nonfusion matmulbias kernel (input size M:1024 N:1024) ...
triton_output: tensor([[  2.7480,  43.6250,  30.9531,  ...,  26.2500,   8.8203,   6.4609],
        [  7.1562, -21.3906,  17.7969,  ..., -16.8125,  29.6562, -48.3125],
        [ 13.8125,  -6.4609,   0.5146,  ...,  48.0000,  25.1562, -27.5156],
        ...,
        [-54.4375,  -1.4375,  -0.4214,  ...,   5.4297,  17.2812, -39.0625],
        [  9.6094, -52.2500,  43.0000,  ..., -45.8438, -49.3125,  32.6875],
        [-12.8516,   9.5391,  53.3750,  ...,  -9.3438,   7.6797,  54.1250]],
       device='cuda:0', dtype=torch.float16)


##### Large Input

In [9]:
!proton -d tree -n profiles/nonfusion_matmulbias_32768x32768 -k triton triton/matmulbias.py --M 32768 --N 32768 nonfusion

Running the nonfusion matmulbias kernel (input size M:32768 N:32768) ...
triton_output: tensor([[ -60.8750,   56.6562,   57.0938,  ...,  274.0000,  295.2500,
           14.8828],
        [ 205.3750, -353.7500, -102.1875,  ...,  -24.4219,   57.0625,
          -30.7969],
        [  56.2188,  258.7500,   46.6562,  ..., -366.0000,  395.2500,
          285.2500],
        ...,
        [  -5.0977,   99.9375,  -73.5625,  ...,  -70.5000,  204.2500,
         -156.8750],
        [  86.8125,  -26.3594, -128.8750,  ...,  -60.7500,  -36.3750,
          100.8750],
        [ -27.6406,   93.5625,   77.9375,  ..., -213.1250, -323.2500,
          -16.8438]], device='cuda:0', dtype=torch.float16)


#### Using the Triton Proton API
Generates a hatchet file, nonfusion_matmulbias_<M>x<N>.hatchet, under the profiles directory and outputs the profile.

##### Small Input

In [10]:
!python triton/matmulbias.py --profile nonfusion

Profiling the nonfusion matmulbias kernel (input size M:1024 N:1024) ...
triton_output: tensor([[  2.7480,  43.6250,  30.9531,  ...,  26.2500,   8.8203,   6.4609],
        [  7.1562, -21.3906,  17.7969,  ..., -16.8125,  29.6562, -48.3125],
        [ 13.8125,  -6.4609,   0.5146,  ...,  48.0000,  25.1562, -27.5156],
        ...,
        [-54.4375,  -1.4375,  -0.4214,  ...,   5.4297,  17.2812, -39.0625],
        [  9.6094, -52.2500,  43.0000,  ..., -45.8438, -49.3125,  32.6875],
        [-12.8516,   9.5391,  53.3750,  ...,  -9.3438,   7.6797,  54.1250]],
       device='cuda:0', dtype=torch.float16)
Proton profile results for nonfusion_matmulbias_1024x1024
1019.654 3635813321220.598 ROOT
├─ 890.020 nan _ZN2at6native29vectorized_elementwise_kernelILi4ENS0_11FillFunctorIiEESt5arrayIPcLm1EEEEviT0_T1_
├─ 44.832 587901839487.468 bias_kernel [M=1024, N=1024] [BM=128, BN=128]
├─ 46.623 555476064829.953 bias_kernel [M=1024, N=1024] [BM=128, BN=256]
└─ 38.180 95732061307624.141 matmul_kernel [M=102

##### Large Input

In [11]:
!python triton/matmulbias.py --M 32768 --N 32768 --profile nonfusion

Profiling the nonfusion matmulbias kernel (input size M:32768 N:32768) ...
triton_output: tensor([[ -60.8750,   56.6562,   57.0938,  ...,  274.0000,  295.2500,
           14.8828],
        [ 205.3750, -353.7500, -102.1875,  ...,  -24.4219,   57.0625,
          -30.7969],
        [  56.2188,  258.7500,   46.6562,  ..., -366.0000,  395.2500,
          285.2500],
        ...,
        [  -5.0977,   99.9375,  -73.5625,  ...,  -70.5000,  204.2500,
         -156.8750],
        [  86.8125,  -26.3594, -128.8750,  ...,  -60.7500,  -36.3750,
          100.8750],
        [ -27.6406,   93.5625,   77.9375,  ..., -213.1250, -323.2500,
          -16.8438]], device='cuda:0', dtype=torch.float16)
Proton profile results for nonfusion_matmulbias_32768x32768
5961.749 201119116701821.125 ROOT
├─ 51.216 nan _ZN2at6native29vectorized_elementwise_kernelILi4ENS0_11FillFunctorIiEESt5arrayIPcLm1EEEEviT0_T1_
├─ 809.386 1716636647566.241 bias_kernel [M=32768, N=32768] [BM=128, BN=128]
├─ 802.369 1699533313948.809 b

#### Using the ROCm Systems Compute profiler
Generates files found under the workloads/nonfusion_matmulbias_<M>x<N>/<AMD GPU Model Name> directory.

##### Small Input

In [12]:
!ROCPROF=rocprofiler-sdk rocprof-compute profile -n nonfusion_matmulbias_1024x1024 -- python triton/matmulbias.py nonfusion


                                 __                                       _
 _ __ ___   ___ _ __  _ __ ___  / _|       ___ ___  _ __ ___  _ __  _   _| |_ ___
| '__/ _ \ / __| '_ \| '__/ _ \| |_ _____ / __/ _ \| '_ ` _ \| '_ \| | | | __/ _ \
| | | (_) | (__| |_) | | | (_) |  _|_____| (_| (_) | | | | | | |_) | |_| | ||  __/
|_|  \___/ \___| .__/|_|  \___/|_|        \___\___/|_| |_| |_| .__/ \__,_|\__\___|
               |_|                                           |_|

WARNING DEPRECATION WARNING: rocm-smi is deprecated in ROCm 7.0 and will be removed from rocprof-compute in ROCm 7.1. Please migrate to amd-smi for compute partition parsing. For migration help, see https://github.com/ROCm/amdsmi
   INFO Rocprofiler-Compute version: 3.2.3
   INFO Profiler choice: rocprofiler-sdk
   INFO Path: /workspace/user/workloads/nonfusion_matmulbias_1024x1024/MI300X_A1
   INFO Target: MI300X_A1
   INFO Command: python triton/matmulbias.py nonfusion
   INFO Kernel Selection: None
   INFO Dispatch Se

##### Large Input

In [13]:
!ROCPROF=rocprofiler-sdk rocprof-compute profile -n nonfusion_matmulbias_32768x32768 -- python triton/matmulbias.py --M 32768 --N 32768 nonfusion


                                 __                                       _
 _ __ ___   ___ _ __  _ __ ___  / _|       ___ ___  _ __ ___  _ __  _   _| |_ ___
| '__/ _ \ / __| '_ \| '__/ _ \| |_ _____ / __/ _ \| '_ ` _ \| '_ \| | | | __/ _ \
| | | (_) | (__| |_) | | | (_) |  _|_____| (_| (_) | | | | | | |_) | |_| | ||  __/
|_|  \___/ \___| .__/|_|  \___/|_|        \___\___/|_| |_| |_| .__/ \__,_|\__\___|
               |_|                                           |_|

WARNING DEPRECATION WARNING: rocm-smi is deprecated in ROCm 7.0 and will be removed from rocprof-compute in ROCm 7.1. Please migrate to amd-smi for compute partition parsing. For migration help, see https://github.com/ROCm/amdsmi
   INFO Rocprofiler-Compute version: 3.2.3
   INFO Profiler choice: rocprofiler-sdk
   INFO Path: /workspace/user/workloads/nonfusion_matmulbias_32768x32768/MI300X_A1
   INFO Target: MI300X_A1
   INFO Command: python triton/matmulbias.py --M 32768 --N 32768 nonfusion
   INFO Kernel Selection: No

### Analyze the Non-Fusion MatMulBias Kernel Profiles

#### Using the Triton Proton Viewer CLI

##### Help

In [14]:
!proton-viewer --help

usage: proton-viewer [-h] [-l] [-m METRICS] [-i INCLUDE] [-e EXCLUDE]
                     [-t THRESHOLD] [-d DEPTH]
                     [-f {full,file_function_line,function_line,file_function}]
                     [--print-sorted] [--diff-profile DIFF_PROFILE]

Performance data viewer for proton profiles.

options:
  -h, --help            show this help message and exit
  -l, --list            List available metrics. Metric names are case insensitive and ignore units.
                        Derived metrics can be created when source metrics are available.
                        - time/s, time/ms, time/us, time/ns: time
                        - avg_time/s, avg_time/ms, avg_time/us, avg_time/ns: time / count
                        - flop[<8/16/32/64>]/s, gflop[<8/16/32/64>]/s, tflop[<8/16/32/64>]/s: flops / time
                        - byte/s, gbyte/s, tbyte/s: bytes / time
                        - util: max(sum(flops<width>) / peak_flops<width>_time, sum(bytes) / peak_bandwid

##### Small Input

In [15]:
!proton-viewer -m time/ms,flops16 profiles/nonfusion_matmulbias_1024x1024.hatchet

1019.654 3707271905280.000 ROOT
├─ 890.020 nan _ZN2at6native29vectorized_elementwise_kernelILi4ENS0_11FillFunctorIiEESt5arrayIPcLm1EEEEviT0_T1_
├─ 44.832 26357006336.000 bias_kernel [M=1024, N=1024] [BM=128, BN=128]
├─ 46.623 25897730048.000 bias_kernel [M=1024, N=1024] [BM=128, BN=256]
└─ 38.180 3655017168896.000 matmul_kernel [M=1024, N=1024, K=1024] [BM=128, BN=128 BK=64]

Legend (Metric: time/ms (inc) Min: 38.18 Max: 1019.65)
█ 921.51 - 1019.65
█ 725.21 - 921.51
█ 528.92 - 725.21
█ 332.62 - 528.92
█ 136.33 - 332.62
█ 38.18 - 136.33

name User code    ◀  Only in left graph    ▶  Only in right graph



##### Large Input

In [16]:
!proton-viewer -m time/ms,flops16 profiles/nonfusion_matmulbias_32768x32768.hatchet

5961.749 1199021725057024.000 ROOT
├─ 51.216 nan _ZN2at6native29vectorized_elementwise_kernelILi4ENS0_11FillFunctorIiEESt5arrayIPcLm1EEEEviT0_T1_
├─ 809.386 1389421920256.000 bias_kernel [M=32768, N=32768] [BM=128, BN=128]
├─ 802.369 1363652116480.000 bias_kernel [M=32768, N=32768] [BM=128, BN=256]
└─ 4298.778 1196268651020288.000 matmul_kernel [M=32768, N=32768, K=32768] [BM=128, BN=128 BK=64]

Legend (Metric: time/ms (inc) Min: 51.22 Max: 5961.75)
█ 5370.70 - 5961.75
█ 4188.59 - 5370.70
█ 3006.48 - 4188.59
█ 1824.38 - 3006.48
█ 642.27 - 1824.38
█ 51.22 - 642.27

name User code    ◀  Only in left graph    ▶  Only in right graph



#### Using the ROCm Systems Compute analyzer

##### Small Input

In [17]:
!rocprof-compute analyze -p workloads/nonfusion_matmulbias_1024x1024/MI300X_A1 -k 1 2


                                 __                                       _
 _ __ ___   ___ _ __  _ __ ___  / _|       ___ ___  _ __ ___  _ __  _   _| |_ ___
| '__/ _ \ / __| '_ \| '__/ _ \| |_ _____ / __/ _ \| '_ ` _ \| '_ \| | | | __/ _ \
| | | (_) | (__| |_) | | | (_) |  _|_____| (_| (_) | | | | | | |_) | |_| | ||  __/
|_|  \___/ \___| .__/|_|  \___/|_|        \___\___/|_| |_| |_| .__/ \__,_|\__\___|
               |_|                                           |_|

   INFO Analysis mode = cli
   INFO [analysis] deriving rocprofiler-compute metrics...
WARNING PC sampling: can not detect pc sampling method without /workspace/user/workloads/nonfusion_matmulbias_1024x1024/MI300X_A1/ps_file_pc_sampling_host_trap.csv 

--------------------------------------------------------------------------------
0. Top Stats
0.1 Top Kernels
╒════╤══════════════════════════════════════════╤═════════╤═════════════╤════════════╤══════════════╤═══════╤═════╕
│    │ Kernel_Name                             

##### Large Input

In [18]:
!rocprof-compute analyze -p workloads/nonfusion_matmulbias_32768x32768/MI300X_A1 -k 0 1


                                 __                                       _
 _ __ ___   ___ _ __  _ __ ___  / _|       ___ ___  _ __ ___  _ __  _   _| |_ ___
| '__/ _ \ / __| '_ \| '__/ _ \| |_ _____ / __/ _ \| '_ ` _ \| '_ \| | | | __/ _ \
| | | (_) | (__| |_) | | | (_) |  _|_____| (_| (_) | | | | | | |_) | |_| | ||  __/
|_|  \___/ \___| .__/|_|  \___/|_|        \___\___/|_| |_| |_| .__/ \__,_|\__\___|
               |_|                                           |_|

   INFO Analysis mode = cli
   INFO [analysis] deriving rocprofiler-compute metrics...
WARNING PC sampling: can not detect pc sampling method without /workspace/user/workloads/nonfusion_matmulbias_32768x32768/MI300X_A1/ps_file_pc_sampling_host_trap.csv 

--------------------------------------------------------------------------------
0. Top Stats
0.1 Top Kernels
╒════╤══════════════════════════════════════════╤═════════╤═══════════════╤══════════════╤══════════════╤═══════╤═════╕
│    │ Kernel_Name                       

## Fusion MatMulBias Kernel

### Verify the Fusion MatMulBias Kernel
This also ensures the kernel has been built and cached. The results from the triton kernel are compared against torch.

#### Small Input

In [19]:
!python triton/matmulbias.py --verify fusion

Running the fusion matmulbias kernel (input size M:1024 N:1024) ...
triton_output: tensor([[  2.7480,  43.6250,  30.9531,  ...,  26.2500,   8.8203,   6.4609],
        [  7.1562, -21.3906,  17.7969,  ..., -16.8125,  29.6562, -48.3125],
        [ 13.8125,  -6.4609,   0.5146,  ...,  48.0000,  25.1562, -27.5156],
        ...,
        [-54.4375,  -1.4375,  -0.4214,  ...,   5.4297,  17.2812, -39.0625],
        [  9.6094, -52.2500,  43.0000,  ..., -45.8438, -49.3125,  32.6875],
        [-12.8516,   9.5391,  53.3750,  ...,  -9.3438,   7.6797,  54.1250]],
       device='cuda:0', dtype=torch.float16)
Running the torch matmulbias kernel ...
torch_output: tensor([[  2.7480,  43.6562,  30.9531,  ...,  26.2500,   8.8203,   6.4609],
        [  7.1602, -21.3906,  17.7969,  ..., -16.8125,  29.6406, -48.3125],
        [ 13.8125,  -6.4609,   0.5142,  ...,  48.0312,  25.1406, -27.5156],
        ...,
        [-54.4375,  -1.4375,  -0.4214,  ...,   5.4297,  17.2812, -39.0625],
        [  9.6094, -52.2500,  4

#### Large Input

In [20]:
!python triton/matmulbias.py --M 32768 --N 32768 --verify fusion

Running the fusion matmulbias kernel (input size M:32768 N:32768) ...
triton_output: tensor([[ -60.8750,   56.6562,   57.0938,  ...,  274.0000,  295.2500,
           14.8828],
        [ 205.3750, -353.7500, -102.1875,  ...,  -24.4219,   57.0625,
          -30.7969],
        [  56.2188,  258.7500,   46.6562,  ..., -366.0000,  395.2500,
          285.2500],
        ...,
        [  -5.0977,   99.9375,  -73.5625,  ...,  -70.5000,  204.2500,
         -156.8750],
        [  86.8125,  -26.3594, -128.8750,  ...,  -60.7500,  -36.3750,
          100.8750],
        [ -27.6406,   93.5625,   77.9375,  ..., -213.1250, -323.2500,
          -16.8438]], device='cuda:0', dtype=torch.float16)
Running the torch matmulbias kernel ...
torch_output: tensor([[ -60.8750,   56.6562,   57.1250,  ...,  274.0000,  295.2500,
           14.8828],
        [ 205.3750, -353.7500, -102.1875,  ...,  -24.4219,   57.0312,
          -30.7969],
        [  56.2188,  258.7500,   46.6562,  ..., -366.0000,  395.2500,
          2

### Profile the Fusion MatMulBias Kernel

#### Using the Triton Proton CLI
Generates a hatchet file, fusion_matmulbias_<M>x<N>.hatchet, under the profiles directory.

##### Small Input

In [21]:
!proton -d tree -n profiles/fusion_matmulbias_1024x1024 -k triton triton/matmulbias.py fusion

Running the fusion matmulbias kernel (input size M:1024 N:1024) ...
triton_output: tensor([[  2.7480,  43.6250,  30.9531,  ...,  26.2500,   8.8203,   6.4609],
        [  7.1562, -21.3906,  17.7969,  ..., -16.8125,  29.6562, -48.3125],
        [ 13.8125,  -6.4609,   0.5146,  ...,  48.0000,  25.1562, -27.5156],
        ...,
        [-54.4375,  -1.4375,  -0.4214,  ...,   5.4297,  17.2812, -39.0625],
        [  9.6094, -52.2500,  43.0000,  ..., -45.8438, -49.3125,  32.6875],
        [-12.8516,   9.5391,  53.3750,  ...,  -9.3438,   7.6797,  54.1250]],
       device='cuda:0', dtype=torch.float16)


##### Large Input

In [22]:
!proton -d tree -n profiles/fusion_matmulbias_32768x32768 -k triton triton/matmulbias.py --M 32768 --N 32768 fusion

Running the fusion matmulbias kernel (input size M:32768 N:32768) ...
triton_output: tensor([[ -60.8750,   56.6562,   57.0938,  ...,  274.0000,  295.2500,
           14.8828],
        [ 205.3750, -353.7500, -102.1875,  ...,  -24.4219,   57.0625,
          -30.7969],
        [  56.2188,  258.7500,   46.6562,  ..., -366.0000,  395.2500,
          285.2500],
        ...,
        [  -5.0977,   99.9375,  -73.5625,  ...,  -70.5000,  204.2500,
         -156.8750],
        [  86.8125,  -26.3594, -128.8750,  ...,  -60.7500,  -36.3750,
          100.8750],
        [ -27.6406,   93.5625,   77.9375,  ..., -213.1250, -323.2500,
          -16.8438]], device='cuda:0', dtype=torch.float16)


#### Using the Triton Proton API
Generates a hatchet file, fusion_matmulbias_<M>x<N>.hatchet, under the profiles directory and outputs the profile.

##### Small Input

In [23]:
!python triton/matmulbias.py --profile fusion

Profiling the fusion matmulbias kernel (input size M:1024 N:1024) ...
triton_output: tensor([[  2.7480,  43.6250,  30.9531,  ...,  26.2500,   8.8203,   6.4609],
        [  7.1562, -21.3906,  17.7969,  ..., -16.8125,  29.6562, -48.3125],
        [ 13.8125,  -6.4609,   0.5146,  ...,  48.0000,  25.1562, -27.5156],
        ...,
        [-54.4375,  -1.4375,  -0.4214,  ...,   5.4297,  17.2812, -39.0625],
        [  9.6094, -52.2500,  43.0000,  ..., -45.8438, -49.3125,  32.6875],
        [-12.8516,   9.5391,  53.3750,  ...,  -9.3438,   7.6797,  54.1250]],
       device='cuda:0', dtype=torch.float16)
Proton profile results for fusion_matmulbias_1024x1024
96.044 37765020802695.430 ROOT
├─ 56.706 nan _ZN2at6native29vectorized_elementwise_kernelILi4ENS0_11FillFunctorIiEESt5arrayIPcLm1EEEEviT0_T1_
└─ 39.338 92202873718740.719 matmulbias_kernel [M=1024, N=1024, K=1024] [BM=128, BN=128 BK=64]

Legend (Metric: time/ms (inc) Min: 39.34 Max: 96.04)
█ 90.37 - 96.04
█ 79.03 - 90.37
█ 67.69 - 79.03
█ 56.3

##### Large Input

In [24]:
!python triton/matmulbias.py --M 32768 --N 32768 --profile fusion

Profiling the fusion matmulbias kernel (input size M:32768 N:32768) ...
triton_output: tensor([[ -60.8750,   56.6562,   57.0938,  ...,  274.0000,  295.2500,
           14.8828],
        [ 205.3750, -353.7500, -102.1875,  ...,  -24.4219,   57.0625,
          -30.7969],
        [  56.2188,  258.7500,   46.6562,  ..., -366.0000,  395.2500,
          285.2500],
        ...,
        [  -5.0977,   99.9375,  -73.5625,  ...,  -70.5000,  204.2500,
         -156.8750],
        [  86.8125,  -26.3594, -128.8750,  ...,  -60.7500,  -36.3750,
          100.8750],
        [ -27.6406,   93.5625,   77.9375,  ..., -213.1250, -323.2500,
          -16.8438]], device='cuda:0', dtype=torch.float16)
Proton profile results for fusion_matmulbias_32768x32768
4268.561 280251057697337.656 ROOT
├─ 0.557 nan _ZN2at6native29vectorized_elementwise_kernelILi4ENS0_11FillFunctorIiEESt5arrayIPcLm1EEEEviT0_T1_
└─ 4268.003 280287657419565.031 matmulbias_kernel [M=32768, N=32768, K=32768] [BM=128, BN=128 BK=64]

Legend (Metr

#### Using the ROCm Systems Compute profiler
Generates files found under the workloads/fusion_matmulbias_<M>x<N>/<AMD GPU Model Name> directory.

##### Small Input

In [25]:
!ROCPROF=rocprofiler-sdk rocprof-compute profile -n fusion_matmulbias_1024x1024 -- python triton/matmulbias.py fusion


                                 __                                       _
 _ __ ___   ___ _ __  _ __ ___  / _|       ___ ___  _ __ ___  _ __  _   _| |_ ___
| '__/ _ \ / __| '_ \| '__/ _ \| |_ _____ / __/ _ \| '_ ` _ \| '_ \| | | | __/ _ \
| | | (_) | (__| |_) | | | (_) |  _|_____| (_| (_) | | | | | | |_) | |_| | ||  __/
|_|  \___/ \___| .__/|_|  \___/|_|        \___\___/|_| |_| |_| .__/ \__,_|\__\___|
               |_|                                           |_|

WARNING DEPRECATION WARNING: rocm-smi is deprecated in ROCm 7.0 and will be removed from rocprof-compute in ROCm 7.1. Please migrate to amd-smi for compute partition parsing. For migration help, see https://github.com/ROCm/amdsmi
   INFO Rocprofiler-Compute version: 3.2.3
   INFO Profiler choice: rocprofiler-sdk
   INFO Path: /workspace/user/workloads/fusion_matmulbias_1024x1024/MI300X_A1
   INFO Target: MI300X_A1
   INFO Command: python triton/matmulbias.py fusion
   INFO Kernel Selection: None
   INFO Dispatch Selectio

##### Large Input

In [26]:
!ROCPROF=rocprofiler-sdk rocprof-compute profile -n fusion_matmulbias_32768x32768 -- python triton/matmulbias.py --M 32768 --N 32768 fusion


                                 __                                       _
 _ __ ___   ___ _ __  _ __ ___  / _|       ___ ___  _ __ ___  _ __  _   _| |_ ___
| '__/ _ \ / __| '_ \| '__/ _ \| |_ _____ / __/ _ \| '_ ` _ \| '_ \| | | | __/ _ \
| | | (_) | (__| |_) | | | (_) |  _|_____| (_| (_) | | | | | | |_) | |_| | ||  __/
|_|  \___/ \___| .__/|_|  \___/|_|        \___\___/|_| |_| |_| .__/ \__,_|\__\___|
               |_|                                           |_|

WARNING DEPRECATION WARNING: rocm-smi is deprecated in ROCm 7.0 and will be removed from rocprof-compute in ROCm 7.1. Please migrate to amd-smi for compute partition parsing. For migration help, see https://github.com/ROCm/amdsmi
   INFO Rocprofiler-Compute version: 3.2.3
   INFO Profiler choice: rocprofiler-sdk
   INFO Path: /workspace/user/workloads/fusion_matmulbias_32768x32768/MI300X_A1
   INFO Target: MI300X_A1
   INFO Command: python triton/matmulbias.py --M 32768 --N 32768 fusion
   INFO Kernel Selection: None
   

### Analyze the Fusion MatMulBias Kernel Profiles

#### Using the Triton Proton

##### Small Input

In [27]:
!proton-viewer -m time/ms,flops16 profiles/fusion_matmulbias_1024x1024.hatchet

96.044 3627099881472.000 ROOT
├─ 56.706 nan _ZN2at6native29vectorized_elementwise_kernelILi4ENS0_11FillFunctorIiEESt5arrayIPcLm1EEEEviT0_T1_
└─ 39.338 3627099881472.000 matmulbias_kernel [M=1024, N=1024, K=1024] [BM=128, BN=128 BK=64]

Legend (Metric: time/ms (inc) Min: 39.34 Max: 96.04)
█ 90.37 - 96.04
█ 79.03 - 90.37
█ 67.69 - 79.03
█ 56.35 - 67.69
█ 45.01 - 56.35
█ 39.34 - 45.01

name User code    ◀  Only in left graph    ▶  Only in right graph



##### Large Input

In [28]:
!proton-viewer -m time/ms,flops16 profiles/fusion_matmulbias_32768x32768.hatchet

4268.561 1196268651020288.000 ROOT
├─ 0.557 nan _ZN2at6native29vectorized_elementwise_kernelILi4ENS0_11FillFunctorIiEESt5arrayIPcLm1EEEEviT0_T1_
└─ 4268.003 1196268651020288.000 matmulbias_kernel [M=32768, N=32768, K=32768] [BM=128, BN=128 BK=64]

Legend (Metric: time/ms (inc) Min: 0.56 Max: 4268.56)
█ 3841.76 - 4268.56
█ 2988.16 - 3841.76
█ 2134.56 - 2988.16
█ 1280.96 - 2134.56
█ 427.36 - 1280.96
█ 0.56 - 427.36

name User code    ◀  Only in left graph    ▶  Only in right graph



#### Using the ROCm Systems Compute analyzer

##### Small Input

In [29]:
!rocprof-compute analyze -p workloads/fusion_matmulbias_1024x1024/MI300X_A1 -k 0


                                 __                                       _
 _ __ ___   ___ _ __  _ __ ___  / _|       ___ ___  _ __ ___  _ __  _   _| |_ ___
| '__/ _ \ / __| '_ \| '__/ _ \| |_ _____ / __/ _ \| '_ ` _ \| '_ \| | | | __/ _ \
| | | (_) | (__| |_) | | | (_) |  _|_____| (_| (_) | | | | | | |_) | |_| | ||  __/
|_|  \___/ \___| .__/|_|  \___/|_|        \___\___/|_| |_| |_| .__/ \__,_|\__\___|
               |_|                                           |_|

   INFO Analysis mode = cli
   INFO [analysis] deriving rocprofiler-compute metrics...
WARNING PC sampling: can not detect pc sampling method without /workspace/user/workloads/fusion_matmulbias_1024x1024/MI300X_A1/ps_file_pc_sampling_host_trap.csv 

--------------------------------------------------------------------------------
0. Top Stats
0.1 Top Kernels
╒════╤══════════════════════════════════════════╤═════════╤════════════╤════════════╤══════════════╤═══════╤═════╕
│    │ Kernel_Name                              │  

##### Large Input

In [30]:
!rocprof-compute analyze -p workloads/fusion_matmulbias_32768x32768/MI300X_A1 -k 0


                                 __                                       _
 _ __ ___   ___ _ __  _ __ ___  / _|       ___ ___  _ __ ___  _ __  _   _| |_ ___
| '__/ _ \ / __| '_ \| '__/ _ \| |_ _____ / __/ _ \| '_ ` _ \| '_ \| | | | __/ _ \
| | | (_) | (__| |_) | | | (_) |  _|_____| (_| (_) | | | | | | |_) | |_| | ||  __/
|_|  \___/ \___| .__/|_|  \___/|_|        \___\___/|_| |_| |_| .__/ \__,_|\__\___|
               |_|                                           |_|

   INFO Analysis mode = cli
   INFO [analysis] deriving rocprofiler-compute metrics...
WARNING PC sampling: can not detect pc sampling method without /workspace/user/workloads/fusion_matmulbias_32768x32768/MI300X_A1/ps_file_pc_sampling_host_trap.csv 

--------------------------------------------------------------------------------
0. Top Stats
0.1 Top Kernels
╒════╤══════════════════════════════════════════╤═════════╤═══════════════╤══════════════╤══════════════╤═══════╤═════╕
│    │ Kernel_Name                          